### making site_sandali excel files in site folder(bahar_sandali_All + electro is needed)
### use_bahar_sandali_full = bahar_sandali_All + electro

In [14]:
"""!git clone https://github.com/ahb45/my_pandas_projects.git
%cd my_pandas_projects
!pip install python-docx docx2pdf"""

'!git clone https://github.com/ahb45/my_pandas_projects.git\n%cd my_pandas_projects\n!pip install python-docx docx2pdf'

In [1]:
import pandas as pd
import os
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment

# مسیر فایل ورودی
file_path = 'bahar_sandali_full.xlsx'

# خواندن فایل اکسل
df = pd.read_excel(file_path)

# ساخت پوشه برای ذخیره فایل‌های خروجی
output_dir = 'site_excell'
os.makedirs(output_dir, exist_ok=True)

# دیکشنری عرض ستون‌ها (برحسب character width)
column_widths = {
    'std': 15,
    'name': 10,
    'fname': 13,
    'code': 10,       # ≈ 100 pixels
    'dars': 30,       # ≈ 300 pixels
    'tarikh': 15,
    'saat': 12,
    'sandali': 8,
    'noe': 10,
    'makan': 15
}

# گروه‌بندی بر اساس tarikh و saat
grouped = df.groupby(['tarikh', 'saat'])

for (tarikh, saat), group in grouped:
    # مرتب‌سازی بر اساس ستون fname
    sorted_group = group.sort_values(by='fname')

    # ساخت نام فایل
    filename = f"{tarikh}_{saat}".replace("/", "-").replace(":", "-") + ".xlsx"
    output_path = os.path.join(output_dir, filename)

    # ذخیره به اکسل
    sorted_group.to_excel(output_path, index=False)

    # اعمال تنظیمات ظاهری با openpyxl
    wb = load_workbook(output_path)
    ws = wb.active

    # وسط‌چین کردن همه سلول‌ها و تنظیم عرض ستون‌ها
    for col_idx, column_cells in enumerate(ws.columns, start=1):
        col_letter = get_column_letter(col_idx)
        header = ws[f'{col_letter}1'].value
        if header in column_widths:
            ws.column_dimensions[col_letter].width = column_widths[header]
        for cell in column_cells:
            cell.alignment = Alignment(horizontal='center', vertical='center')

    wb.save(output_path)
print("Excell files created exesfuliy")



Excell files created exesfuliy


###  make word files Distinct

In [2]:
import pandas as pd
import os
from docx import Document
from docx.shared import Pt, Inches
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH

def set_repeat_table_header(row):
    tr = row._tr
    trPr = tr.get_or_add_trPr()
    tblHeader = OxmlElement('w:tblHeader')
    trPr.append(tblHeader)

def set_cell_background(cell, color_hex):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd = OxmlElement('w:shd')
    shd.set(qn('w:val'), 'clear')
    shd.set(qn('w:color'), 'auto')
    shd.set(qn('w:fill'), color_hex)
    tcPr.append(shd)

def prevent_row_split(row):
    tr = row._tr
    trPr = tr.get_or_add_trPr()
    cant_split = OxmlElement('w:cantSplit')
    trPr.append(cant_split)
def set_cell_alignment_right(cell):
    for paragraph in cell.paragraphs:
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        pPr = paragraph._element.get_or_add_pPr()
        bidi = OxmlElement('w:bidi')
        bidi.set(qn('w:val'), '1')
        pPr.append(bidi)

def generate_doc(group, output_path, column_map, display_columns, column_widths_inch, tarikh, saat):
    doc = Document()

    # تنظیم فونت پیش‌فرض
    style = doc.styles['Normal']
    style.font.name = 'B Nazanin'
    style._element.rPr.rFonts.set(qn('w:eastAsia'), 'B Nazanin')
    style.font.size = Pt(12)

    # افزودن تاریخ و ساعت در هدر همه صفحات
    section = doc.sections[0]
    header = section.header
    header_para = header.paragraphs[0] if header.paragraphs else header.add_paragraph()
    header_para.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = header_para.add_run(f"تاریخ: {tarikh}    ساعت: {saat}")
    run.font.name = 'B Nazanin'
    run.font.size = Pt(14)
    run.bold = True
    pPr = header_para._element.get_or_add_pPr()
    bidi = OxmlElement('w:bidi')
    bidi.set(qn('w:val'), '1')
    pPr.append(bidi)

    doc.add_paragraph("")  # فاصله قبل جدول

    table = doc.add_table(rows=1, cols=len(display_columns))
    table.style = 'Table Grid'
    table.alignment = WD_TABLE_ALIGNMENT.RIGHT

    # هدر جدول
    header_row = table.rows[0]
    for i, col in enumerate(display_columns):
        cell = header_row.cells[i]
        cell.text = column_map[col]
        set_cell_background(cell, 'D9D9D9')
        set_cell_alignment_right(cell)
        cell.width = Inches(column_widths_inch.get(col, 1.2))
    set_repeat_table_header(header_row)
    prevent_row_split(header_row)

    # مرتب‌سازی بر اساس نام خانوادگی
    group_sorted = group.sort_values(by='fname', kind='stable')

    # ردیف‌های جدول
    for idx, (_, row_data) in enumerate(group_sorted.iterrows()):
        row = table.add_row()
        prevent_row_split(row)
        bg_color = 'FFFFFF' if idx % 2 == 0 else 'F2F2F2'
        for i, col in enumerate(display_columns):
            cell = row.cells[i]
            set_cell_background(cell, bg_color)
            set_cell_alignment_right(cell)
            cell.width = Inches(column_widths_inch.get(col, 1.2))
            paragraph = cell.paragraphs[0]
            run = paragraph.add_run(str(row_data[col]))
            if col == 'sandali':
                run.bold = True
            run.font.name = 'B Nazanin'
            run.font.size = Pt(13)

    # ردیف خالی پایانی
    row = table.add_row()
    prevent_row_split(row)
    for cell in row.cells:
        cell.text = ""
        set_cell_alignment_right(cell)

    # اعمال راست‌چین بودن برای سایر پاراگراف‌ها
    for paragraph in doc.paragraphs:
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        pPr = paragraph._element.get_or_add_pPr()
        bidi = OxmlElement('w:bidi')
        bidi.set(qn('w:val'), '1')
        pPr.append(bidi)

    doc.save(output_path)

# مسیرها و تنظیمات اولیه
file_path = 'bahar_sandali_full.xlsx'
site_output_dir = 'word_electro'
other_output_dir = 'word_pre'
os.makedirs(site_output_dir, exist_ok=True)
os.makedirs(other_output_dir, exist_ok=True)

column_map = {
    'std': 'شماره دانشجویی',
    'fname': 'نام خانوادگی',
    'name': 'نام',
    'code': 'کد درس',
    'tarikh': 'تاریخ',
    'saat': 'ساعت',
    'sandali': 'شماره صندلی',
    'makan': 'مکان',
}
display_columns = [c for c in column_map if c not in ['tarikh', 'saat']]

column_widths_px = {
    'std': 100,
    'fname': 130,
    'name': 60,
    'code': 80,
    'dars': 360,
    'sandali': 65,
    'noe': 100,
    'makan': 210
}
column_widths_inch = {k: v / 96 for k, v in column_widths_px.items()}

# بارگذاری داده‌ها
df = pd.read_excel(file_path)
df = df[[c for c in column_map.keys()]]

# گروه‌بندی بر اساس تاریخ و ساعت
grouped = df.groupby(['tarikh', 'saat'])

for (tarikh, saat), group in grouped:
    group_site = group[group['makan'] == 'سايت']
    group_other = group[group['makan'] != 'سايت']

    safe_name = f"{tarikh}_{saat}".replace("/", "-").replace(":", "-")

    if not group_site.empty:
        output_path = os.path.join(site_output_dir, f"{safe_name}_site.docx")
        generate_doc(group_site, output_path, column_map, display_columns, column_widths_inch, tarikh, saat)

    if not group_other.empty:
        output_path = os.path.join(other_output_dir, f"{safe_name}.docx")
        generate_doc(group_other, output_path, column_map, display_columns, column_widths_inch, tarikh, saat)

print("✅ فایل‌های مخصوص سایت و دیگر مکان‌ها همراه با تاریخ و ساعت و مرتب‌سازی بر اساس نام خانوادگی ساخته شدند.")


✅ فایل‌های مخصوص سایت و دیگر مکان‌ها همراه با تاریخ و ساعت و مرتب‌سازی بر اساس نام خانوادگی ساخته شدند.


### convert to pdf

In [31]:
import os
from docx2pdf import convert

# مسیر ورودی و خروجی
input_dir = 'word_electro'
output_dir = 'pdf_electro'

# ساخت پوشه خروجی در صورت نبود
os.makedirs(output_dir, exist_ok=True)

# پیمایش فایل‌ها در پوشه site
for filename in os.listdir(input_dir):
    if filename.endswith('.docx'):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename.replace('.docx', '.pdf'))
        convert(input_path, output_path)

print("تبدیل همه فایل‌های Word به PDF با موفقیت انجام شد.")
# مسیر ورودی و خروجی
input_dir = 'word_pre'
output_dir = 'pdf_pre'

# ساخت پوشه خروجی در صورت نبود
os.makedirs(output_dir, exist_ok=True)

# پیمایش فایل‌ها در پوشه site
for filename in os.listdir(input_dir):
    if filename.endswith('.docx'):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename.replace('.docx', '.pdf'))
        convert(input_path, output_path)

print("تبدیل همه فایل‌های Word به PDF با موفقیت انجام شد.")

100%|██████████| 1/1 [00:04<00:00,  4.03s/it]


تبدیل همه فایل‌های Word به PDF با موفقیت انجام شد.


100%|██████████| 1/1 [00:08<00:00,  8.66s/it]

تبدیل همه فایل‌های Word به PDF با موفقیت انجام شد.


### zip files - just excell and word files- no pdf

In [32]:
import zipfile
import os

folders_to_zip = ['pdf_pre', 'pdf_electro']

for input_dir in folders_to_zip:
    zip_filename = f"{input_dir}.zip"
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(input_dir):
            for file in files:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, start=input_dir)
                zipf.write(full_path, arcname=os.path.join(os.path.basename(input_dir), arcname))
    print(f"✅ پوشه {input_dir} در فایل {zip_filename} ذخیره شد.")


✅ پوشه pdf_pre در فایل pdf_pre.zip ذخیره شد.
✅ پوشه pdf_electro در فایل pdf_electro.zip ذخیره شد.


### colab zip

In [33]:
"""import zipfile
import os
from google.colab import files

folders_to_zip = ['word_electro', 'word_pre','site_excell']

for input_dir in folders_to_zip:
    zip_filename = f"{input_dir}.zip"
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_in_dir in os.walk(input_dir):
            for file in files_in_dir:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, start=input_dir)
                zipf.write(full_path, arcname=os.path.join(os.path.basename(input_dir), arcname))
    print(f"✅ پوشه {input_dir} در فایل {zip_filename} ذخیره شد.")

    # فایل ZIP را برای دانلود در دسترس قرار بده
    files.download(zip_filename)"""


'import zipfile\nimport os\nfrom google.colab import files\n\nfolders_to_zip = [\'word_electro\', \'word_pre\',\'site_excell\']\n\nfor input_dir in folders_to_zip:\n    zip_filename = f"{input_dir}.zip"\n    with zipfile.ZipFile(zip_filename, \'w\', zipfile.ZIP_DEFLATED) as zipf:\n        for root, dirs, files_in_dir in os.walk(input_dir):\n            for file in files_in_dir:\n                full_path = os.path.join(root, file)\n                arcname = os.path.relpath(full_path, start=input_dir)\n                zipf.write(full_path, arcname=os.path.join(os.path.basename(input_dir), arcname))\n    print(f"✅ پوشه {input_dir} در فایل {zip_filename} ذخیره شد.")\n\n    # فایل ZIP را برای دانلود در دسترس قرار بده\n    files.download(zip_filename)'